# SQL-on-FHIR analytics with `pysof` &mdash; the #650 do-nothing baseline

HFS already turns FHIR into a table (`$sql-run` / `$sql-export` / `pysof`). This
notebook is the **"invest in the client library, bundle no notebook"** option
from the [#650 evaluation](../../docs/sql-on-fhir-analytics-evaluation.md): a
user analyzes SQL-on-FHIR output in their own Jupyter, HFS stays a pure data
source.

Two consumption paths:
- **Path A** &mdash; pull results from a **live HFS** over HTTP as **Arrow** (fast, zero-copy to pandas).
- **Path B** &mdash; run the ViewDefinition **offline** with `pysof`, no server at all.


In [ ]:
import json
import pandas as pd

HFS = "http://localhost:8080"          # a running `cargo run --bin hfs`
TENANT = ""                              # set if multi-tenancy/auth is enabled

view = json.load(open("view.json"))
view


## Path A &mdash; live HFS `$sql-run`, Arrow over HTTP

Start a server (`cargo run --bin hfs`) and seed a few Patients first.

In [ ]:
import pyarrow as pa
import requests

df = None
try:
    headers = {"Content-Type": "application/json",
               "Accept": "application/vnd.apache.arrow.stream"}
    if TENANT:
        headers["X-Tenant-ID"] = TENANT
    r = requests.post(f"{HFS}/$sql-run?_format=arrow",
                      headers=headers, data=json.dumps(view), timeout=10)
    r.raise_for_status()
    table = pa.ipc.open_stream(pa.py_buffer(r.content)).read_all()
    df = table.to_pandas()
    print(f"{len(df)} rows from live $sql-run (Arrow)")
    display(df.head())
except Exception as e:
    print("Live HFS not reachable -- start `cargo run --bin hfs` and seed "
          "Patients. Using Path B (pysof offline) below instead.\n ", e)


In [ ]:
import matplotlib.pyplot as plt

if df is not None and len(df):
    counts = df["gender"].value_counts()
    ax = counts.plot.bar(color="#2a78d6", figsize=(5, 3))
    ax.set_title("Patients by gender (live $sql-run)")
    ax.set_ylabel("count")
    plt.tight_layout(); plt.show()
    print("Total patients:", len(df))
else:
    print("(Path A produced no data -- see Path B)")


## Path B &mdash; `pysof`, fully offline (no server)

`pysof` is the same Rust SOF engine in-process. It runs a ViewDefinition over a
FHIR Bundle with **no HFS server** &mdash; ideal for local analysis, notebooks,
and CI.


In [ ]:
dfb = None
try:
    import pysof
    bundle = json.load(open("sample_bundle.json"))
    # run_view_definition(view, bundle, format) -> bytes; "json" -> a JSON array
    out = pysof.run_view_definition(view, bundle, "json")
    dfb = pd.DataFrame(json.loads(out))
    print(f"pysof {pysof.get_version()} -> {len(dfb)} rows (offline, no server)")
    display(dfb)
except ImportError:
    print("pysof not installed. `uv pip install pysof`  (or, from the repo, "
          "`cd crates/pysof && maturin develop --release`).")
except Exception as e:
    print("pysof call failed -- check the API with help(pysof):", e)


In [ ]:
if dfb is not None and len(dfb):
    counts = dfb["gender"].value_counts()
    ax = counts.plot.bar(color="#1f7a1f", figsize=(5, 3))
    ax.set_title("Patients by gender (pysof, offline)")
    ax.set_ylabel("count")
    plt.tight_layout(); plt.show()


## Takeaway

Nothing was embedded in HFS. The server produced governed, analysis-ready output
(Arrow over `$sql-run`, or `pysof` in-process) and analysis happened in a
standard notebook. This is the **floor** #650 evaluates: cheap, immediately
useful to every Python user, and worth doing regardless of whether the native
in-product notebook (the primary recommendation) is built.
